# ClinicalTrials.gov Node Demo

This notebook follows the dataset notebook contract v1 for the ctgov dataset node.
It validates health, data contracts, RPC, and MCP usage on a running stack.

## Section A: Setup and environment preflight

In [ ]:
import json
import os
import importlib
from typing import Any, Dict, List

required_packages = {
    "requests": "requests",
    "pandas": "pandas",
}
missing = [pip_name for module_name, pip_name in required_packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Install them and rerun this cell.")

import pandas as pd
import requests

# Primary path: orchestrator /ctgov (redirected by orchestrator to /jupyter/ctgov).
# Fallbacks support local compose and direct jupyter proxy access.
NODE_BASE_URL = os.getenv("NODE_BASE_URL", "").strip()
ORCHESTRATOR_URL = os.getenv("ORCHESTRATOR_URL", "").strip()
ORCHESTRATOR_BASE_URL = os.getenv("ORCHESTRATOR_BASE_URL", "").strip()
JUPYTER_PROXY_BASE_URL = os.getenv("JUPYTER_PROXY_BASE_URL", "").strip()
JUPYTER_NODE_URL = os.getenv("JUPYTER_NODE_URL", "http://localhost:8002").strip()

if not NODE_BASE_URL:
    if ORCHESTRATOR_URL:
        NODE_BASE_URL = ORCHESTRATOR_URL.rstrip("/") + "/ctgov"
    elif ORCHESTRATOR_BASE_URL:
        NODE_BASE_URL = ORCHESTRATOR_BASE_URL.rstrip("/") + "/ctgov"
    elif JUPYTER_PROXY_BASE_URL:
        NODE_BASE_URL = JUPYTER_PROXY_BASE_URL.rstrip("/") + "/ctgov"
    else:
        NODE_BASE_URL = JUPYTER_NODE_URL.rstrip("/") + "/ctgov"

print("NODE_BASE_URL=", NODE_BASE_URL)
print("ORCHESTRATOR_URL=", ORCHESTRATOR_URL or "<not set>")
print("ORCHESTRATOR_BASE_URL=", ORCHESTRATOR_BASE_URL or "<not set>")
print("JUPYTER_PROXY_BASE_URL=", JUPYTER_PROXY_BASE_URL or "<not set>")
print("JUPYTER_NODE_URL=", JUPYTER_NODE_URL or "<not set>")


In [ ]:
SESSION = requests.Session()

def api_get(path: str, **params):
    url = f"{NODE_BASE_URL.rstrip('/')}" + path
    return SESSION.get(url, params=params or None, timeout=20)

def api_post(path: str, payload: Dict[str, Any]):
    url = f"{NODE_BASE_URL.rstrip('/')}" + path
    return SESSION.post(url, json=payload, timeout=30)

def assert_ok(response, context: str):
    if response.status_code != 200:
        raise AssertionError(f"{context} failed with {response.status_code}: {response.text[:500]}")

def preview_records(records: List[Dict[str, Any]], n: int = 5):
    rows = records[:n] if isinstance(records, list) else []
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("No records to preview.")

print("Helper functions loaded.")


## Section B: Health and readiness

In [ ]:
health = api_get("/health")
ready = api_get("/ready")

print("/health status:", health.status_code)
print(json.dumps(health.json(), indent=2) if health.headers.get('content-type', '').startswith('application/json') else health.text[:500])
print("/ready status:", ready.status_code)
print(json.dumps(ready.json(), indent=2) if ready.headers.get('content-type', '').startswith('application/json') else ready.text[:500])

assert_ok(health, "health check")
assert_ok(ready, "readiness check")
print("Section B PASS: both endpoints returned 200.")

## Section C: Data discovery contract

In [ ]:
summary = api_get("/data/summary")
tables = api_get("/data/tables")

# tables endpoint is the canonical contract source; keep this strict.
assert_ok(tables, "data tables")

tables_body = tables.json()

if summary.status_code == 200:
    summary_body = summary.json()
else:
    print("/data/summary status:", summary.status_code)
    print((summary.text or "")[:500])
    # Production can be temporarily unseeded; synthesize a contract-compatible
    # summary from /data/tables so the rest of the notebook can continue.
    table_names = sorted((tables_body.get("tables") or {}).keys())
    summary_body = {
        "source": "AACT local mirror",
        "tables": [{"table": t, "row_count": None} for t in table_names],
        "notes": {
            "degraded": "summary endpoint unavailable; using /data/tables fallback",
        },
    }

print("/data/summary keys:", list(summary_body.keys()))
table_list = summary_body.get("tables", [])
display(pd.DataFrame(table_list))

table_schemas = tables_body.get("tables", {})
if table_schemas:
    first_table = sorted(table_schemas.keys())[0]
    print("First schema table:", first_table)
    print("First schema fields:", list(table_schemas[first_table].keys())[:15])
else:
    print("No table schema payload returned.")


## Section D: Query and rejection behavior

In [ ]:
safe_query = api_post("/data/query", {"sql": "SELECT nct_id, status, start_date FROM ctgov_studies LIMIT 5"})
print("safe query status:", safe_query.status_code)
if safe_query.status_code == 200:
    safe_rows = safe_query.json().get("rows", [])
    print("safe query row count:", len(safe_rows))
    preview_records(safe_rows)
else:
    print(safe_query.text[:500])

bad_query = api_post("/data/query", {"sql": "DELETE FROM ctgov_studies"})
print("mutating query status:", bad_query.status_code)
print(bad_query.text[:500])
if bad_query.status_code in (400, 405, 422):
    print("Mutation rejection confirmed.")
else:
    raise AssertionError("Expected mutating query rejection (400/405/422).")

## Section E: A2A and RPC interaction

In [ ]:
a2a = api_get("/.well-known/a2a")
assert_ok(a2a, "a2a card")
print("A2A card excerpt:")
print(json.dumps(a2a.json(), indent=2)[:2000])

minimal_prompt = "List one trial status available in this dataset."
scoped_prompt = "Summarize current trial counts by status and include the top 3 statuses by count."

for prompt in (minimal_prompt, scoped_prompt):
    payload = {
        "message": {
            "role": "user",
            "parts": [{"type": "text", "text": prompt}]
        }
    }
    rpc = api_post("/rpc", payload)
    print("Prompt:", prompt)
    print("RPC status:", rpc.status_code)
    body = rpc.json() if rpc.headers.get('content-type', '').startswith('application/json') else {"text": rpc.text[:1000]}
    print(json.dumps(body, indent=2)[:2000])

## Section F: MCP tool invocation

In [ ]:
tool_name = "search_trials"
tool_args = {"condition": "cancer", "limit": 5}

tools_resp = api_get("/mcp/tools")
if tools_resp.status_code != 200:
    print("MCP tools endpoint unavailable:", tools_resp.status_code)
    print((tools_resp.text or "")[:500])
else:
    tools_payload = tools_resp.json() if tools_resp.headers.get("content-type", "").startswith("application/json") else {}
    available = {t.get("name") for t in tools_payload.get("tools", []) if isinstance(t, dict)}
    print("MCP tool count:", len(available))
    if tool_name not in available:
        print(f"Tool '{tool_name}' not exposed; skipping MCP call.")
    else:
        mcp = api_post("/mcp/call", {"tool": tool_name, "args": tool_args})
        print("tool:", tool_name)
        print("args:", tool_args)
        print("status:", mcp.status_code)
        if mcp.status_code == 200:
            body = mcp.json()
            rows = body.get("rows", []) if isinstance(body, dict) else []
            print("result rows:", len(rows))
            preview_records(rows)
        else:
            print((mcp.text or "")[:1000])


## Section G: Optional orchestrator visibility

In [ ]:
if not ORCHESTRATOR_BASE_URL:
    print("ORCHESTRATOR_BASE_URL not configured; skipping visibility check.")
else:
    candidates = ["/registry/nodes", "/nodes", "/api/nodes"]
    found = False
    for path in candidates:
        try:
            resp = requests.get(f"{ORCHESTRATOR_BASE_URL.rstrip('/')}" + path, timeout=15)
        except Exception as exc:
            print(f"{path}: request failed ({exc})")
            continue

        print(f"{path}: status {resp.status_code}")
        if resp.status_code != 200:
            continue

        body = resp.json() if resp.headers.get('content-type', '').startswith('application/json') else {}
        nodes = body.get("nodes", body if isinstance(body, list) else [])
        text = json.dumps(nodes)[:20000].lower()
        found = "ctgov" in text
        print("ctgov presence detected:", found)
        break

    if not found:
        print("ctgov node not detected from checked orchestrator endpoints.")

## Section H: Troubleshooting and rerun guide

Common failures and quick remediation:

1. 404 route mismatch
- Cause: wrong NODE_BASE_URL or incompatible node image.
- Fix: verify NODE_BASE_URL and confirm expected routes with /health and /.well-known/a2a.

2. 5xx startup race
- Cause: database or node service still initializing.
- Fix: wait 10-30 seconds, re-run Section B, then re-run failed cells.

3. Environment misconfiguration
- Cause: missing NODE_BASE_URL or incorrect ORCHESTRATOR_BASE_URL.
- Fix: export correct values and restart notebook kernel before rerunning.

Rerun order:
- Run Section A once after setting env vars.
- Re-run Sections B-F in order.
- Run Section G only if orchestrator validation is needed.